In [5]:
# =========================================================
# Sentiment Analysis using LSTM
# Text Summarization using DistilBART
# =========================================================

import csv
import re
import string
import warnings
import pickle

import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# =========================================================
# 1. General settings
# =========================================================

warnings.filterwarnings("ignore")

CSV_FILE_PATH = "Reviews.csv"
MAX_VALID_ROWS = 100000

VOCAB_SIZE = 20000
EMBEDDING_DIM = 128
LSTM_UNITS = 64

EPOCHS = 12
BATCH_SIZE = 128


# =========================================================
# 2. Download NLTK packages
# =========================================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# =========================================================
# 3. Prepare preprocessing tools
# =========================================================

stop_words = set(
    stopwords.words("english")
)

# Keep negation words because they change sentiment
negation_words = {
    "not",
    "no",
    "nor",
    "never"
}

stop_words = stop_words.difference(
    negation_words
)

lemmatizer = WordNetLemmatizer()


# =========================================================
# 4. Safe CSV reading function
# =========================================================

def read_csv_safely(file_path, max_valid_rows=100000):
    """
    Read a CSV file and skip corrupted rows.

    Handles:
    - Extra columns
    - Missing columns
    - Broken quotation marks
    - Invalid characters
    """

    valid_rows = []
    bad_rows = []

    with open(
        file_path,
        mode="r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as file:

        reader = csv.reader(
            file,
            delimiter=",",
            quotechar='"',
            strict=True
        )

        try:
            columns = next(reader)

        except StopIteration:
            raise ValueError(
                "The CSV file is empty."
            )

        columns = [
            column.strip().replace("\ufeff", "")
            for column in columns
        ]

        expected_columns = len(columns)

        while len(valid_rows) < max_valid_rows:

            try:
                row = next(reader)

            except StopIteration:
                break

            except csv.Error as error:

                bad_rows.append({
                    "line": reader.line_num,
                    "error": str(error)
                })

                continue

            if len(row) != expected_columns:

                bad_rows.append({
                    "line": reader.line_num,
                    "error": (
                        f"Expected {expected_columns} columns, "
                        f"but found {len(row)} columns."
                    )
                })

                continue

            valid_rows.append(row)

    dataframe = pd.DataFrame(
        valid_rows,
        columns=columns
    )

    return dataframe, bad_rows


# =========================================================
# 5. Load the dataset
# =========================================================

print("\nReading dataset...")

df, bad_rows = read_csv_safely(
    file_path=CSV_FILE_PATH,
    max_valid_rows=MAX_VALID_ROWS
)

print("Dataset loaded successfully.")
print("Initial dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())
print("Skipped corrupted rows:", len(bad_rows))

if bad_rows:

    print("\nFirst corrupted rows:")

    for bad_row in bad_rows[:10]:
        print(bad_row)


# =========================================================
# 6. Check required columns
# =========================================================

required_columns = {
    "Text",
    "Summary",
    "Score"
}

missing_columns = required_columns.difference(
    df.columns
)

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )


# Keep only the required columns
df = df[
    ["Text", "Summary", "Score"]
].copy()


# =========================================================
# 7. Clean Score column
# =========================================================

df["Score"] = pd.to_numeric(
    df["Score"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "Text",
        "Summary",
        "Score"
    ]
)

# Keep only valid scores
df = df[
    df["Score"].isin([1, 2, 3, 4, 5])
].copy()

df["Score"] = df["Score"].astype(int)

print(
    "\nDataset shape after cleaning:",
    df.shape
)


# =========================================================
# 8. Convert score into sentiment
# =========================================================

def map_score_to_sentiment(score):
    """
    1 or 2 = negative
    3 = neutral
    4 or 5 = positive
    """

    if score in [1, 2]:
        return "negative"

    elif score == 3:
        return "neutral"

    else:
        return "positive"


df["Sentiment"] = df["Score"].apply(
    map_score_to_sentiment
)

print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())


# =========================================================
# 9. Text preprocessing functions
# =========================================================

def clean_html_tags(text):
    """
    Remove HTML tags and extra spaces.
    """

    text = str(text)

    text = re.sub(
        r"<[^>]*>",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def normalize_negations(text):
    """
    Expand contractions before punctuation removal.

    Examples:
    don't -> do not
    isn't -> is not
    can't -> can not
    """

    text = str(text).replace("’", "'")

    contractions = {
        "can't": "can not",
        "cannot": "can not",
        "won't": "will not",
        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "hasn't": "has not",
        "haven't": "have not",
        "hadn't": "had not",
        "shouldn't": "should not",
        "wouldn't": "would not",
        "couldn't": "could not",
        "mustn't": "must not",
        "needn't": "need not",
        "mightn't": "might not",
        "shan't": "shall not"
    }

    for contraction, expanded_form in contractions.items():

        text = re.sub(
            rf"\b{re.escape(contraction)}\b",
            expanded_form,
            text,
            flags=re.IGNORECASE
        )

    return text


def preprocess_text(text):
    """
    Complete preprocessing pipeline.
    """

    text = str(text).lower()

    # Expand contractions
    text = normalize_negations(text)

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    tokens = word_tokenize(text)

    cleaned_tokens = []

    for word in tokens:

        if not word.isalpha():
            continue

        if word in stop_words:
            continue

        lemma = lemmatizer.lemmatize(
            word
        )

        cleaned_tokens.append(
            lemma
        )

    return " ".join(cleaned_tokens)


# =========================================================
# 10. Apply preprocessing
# =========================================================

print("\nCleaning review text...")

df["Cleaned_Text"] = df["Text"].apply(
    clean_html_tags
)

df["Processed_Text"] = df["Cleaned_Text"].apply(
    preprocess_text
)

# Remove empty processed text
df = df[
    df["Processed_Text"].str.strip() != ""
].copy()

df = df.reset_index(
    drop=True
)

print("Text preprocessing completed.")
print("Final dataset shape:", df.shape)


# Display examples
print("\nPreprocessing examples:")

for index in range(
    min(3, len(df))
):

    print("\nOriginal text:")
    print(df.loc[index, "Text"])

    print("\nProcessed text:")
    print(df.loc[index, "Processed_Text"])

    print("-" * 60)


# =========================================================
# 11. Split the dataset
# =========================================================

X = df["Processed_Text"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_validation, y_train, y_validation = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.15,
        random_state=42,
        stratify=y_train
    )
)

print("\nDataset split:")
print("Training records:", len(X_train))
print("Validation records:", len(X_validation))
print("Testing records:", len(X_test))


# =========================================================
# 12. Encode sentiment labels
# =========================================================

label_encoder = LabelEncoder()

label_encoder.fit(
    df["Sentiment"]
)

y_train_encoded = label_encoder.transform(
    y_train
)

y_validation_encoded = label_encoder.transform(
    y_validation
)

y_test_encoded = label_encoder.transform(
    y_test
)

print("\nSentiment classes:")

for class_index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{class_index} = {class_name}"
    )


NUMBER_OF_CLASSES = len(
    label_encoder.classes_
)


# =========================================================
# 13. Tokenize the text
# =========================================================

text_tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

# Fit only on training data
text_tokenizer.fit_on_texts(
    X_train.tolist()
)

X_train_sequences = (
    text_tokenizer.texts_to_sequences(
        X_train.tolist()
    )
)

X_validation_sequences = (
    text_tokenizer.texts_to_sequences(
        X_validation.tolist()
    )
)

X_test_sequences = (
    text_tokenizer.texts_to_sequences(
        X_test.tolist()
    )
)


# =========================================================
# 14. Calculate sequence maximum length
# =========================================================

training_lengths = (
    X_train
    .str
    .split()
    .apply(len)
)

MAX_LENGTH = int(
    np.percentile(
        training_lengths,
        95
    )
)

# Keep sequence length between 20 and 250
MAX_LENGTH = max(
    20,
    min(MAX_LENGTH, 250)
)

print(
    "\nMaximum sequence length:",
    MAX_LENGTH
)


# =========================================================
# 15. Apply padding
# =========================================================

X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_validation_padded = pad_sequences(
    X_validation_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("\nPadded data shapes:")
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_validation_padded.shape)
print("Testing shape:", X_test_padded.shape)


# =========================================================
# 16. Calculate class weights
# =========================================================

classes = np.unique(
    y_train_encoded
)

calculated_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_encoded
)

class_weights = {
    int(class_number): float(weight)

    for class_number, weight in zip(
        classes,
        calculated_weights
    )
}

print("\nClass weights:")
print(class_weights)


# =========================================================
# 17. Build the LSTM model
# =========================================================

sentiment_model = Sequential([
    # Explicit input layer
    Input(
        shape=(MAX_LENGTH,),
        name="input_layer"
    ),

    # Convert each word index into a vector
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        name="embedding_layer"
    ),

    # Read the review sequentially
    LSTM(
        units=LSTM_UNITS,
        dropout=0.20,
        name="lstm_layer"
    ),

    # Fully connected layer
    Dense(
        units=64,
        activation="relu",
        name="dense_hidden_layer"
    ),

    # Reduce overfitting
    Dropout(
        rate=0.30,
        name="dropout_layer"
    ),

    # Output layer
    Dense(
        units=NUMBER_OF_CLASSES,
        activation="softmax",
        name="sentiment_output"
    )
])


# =========================================================
# 18. Compile the LSTM model
# =========================================================

sentiment_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================================================
# 19. Display model structure BEFORE training
# =========================================================

print("\n")
print("=" * 70)
print("LSTM MODEL STRUCTURE")
print("=" * 70)

sentiment_model.summary()

print("=" * 70)


# =========================================================
# 20. Define training callbacks
# =========================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_learning_rate = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=0.00001,
    verbose=1
)


# =========================================================
# 21. Train the LSTM model
# =========================================================

print("\nStarting LSTM training...")

history = sentiment_model.fit(
    X_train_padded,
    y_train_encoded,

    validation_data=(
        X_validation_padded,
        y_validation_encoded
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        reduce_learning_rate
    ],

    verbose=1
)


# =========================================================
# 22. Evaluate the model
# =========================================================

print("\nEvaluating the LSTM model...")

test_loss, test_accuracy = sentiment_model.evaluate(
    X_test_padded,
    y_test_encoded,
    verbose=1
)

test_probabilities = sentiment_model.predict(
    X_test_padded,
    verbose=1
)

y_pred_encoded = np.argmax(
    test_probabilities,
    axis=1
)

calculated_accuracy = accuracy_score(
    y_test_encoded,
    y_pred_encoded
)

detailed_report = classification_report(
    y_test_encoded,
    y_pred_encoded,
    labels=list(
        range(NUMBER_OF_CLASSES)
    ),
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
)

confusion_matrix_result = confusion_matrix(
    y_test_encoded,
    y_pred_encoded
)


# =========================================================
# 23. Predict sentiment function
# =========================================================

def predict_sentiment(raw_text):
    """
    Predict the sentiment of a single review.

    Returns:
    - Sentiment
    - Confidence
    - All class probabilities
    """

    if raw_text is None:

        raise ValueError(
            "Input text cannot be None."
        )

    raw_text = str(raw_text).strip()

    if not raw_text:

        raise ValueError(
            "Input text cannot be empty."
        )

    cleaned_text = clean_html_tags(
        raw_text
    )

    processed_text = preprocess_text(
        cleaned_text
    )

    if not processed_text:

        raise ValueError(
            "The text became empty after preprocessing."
        )

    sequence = text_tokenizer.texts_to_sequences(
        [processed_text]
    )

    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    probabilities = sentiment_model.predict(
        padded_sequence,
        verbose=0
    )[0]

    predicted_class_index = int(
        np.argmax(probabilities)
    )

    predicted_sentiment = (
        label_encoder.inverse_transform(
            [predicted_class_index]
        )[0]
    )

    confidence = float(
        probabilities[
            predicted_class_index
        ]
    )

    all_probabilities = {
        class_name: float(
            probabilities[index]
        )

        for index, class_name in enumerate(
            label_encoder.classes_
        )
    }

    return (
        predicted_sentiment,
        confidence,
        all_probabilities
    )


# =========================================================
# 24. Load DistilBART summarization model
# =========================================================

print("\nLoading summarization model...")

SUMMARIZATION_MODEL_NAME = (
    "sshleifer/distilbart-cnn-12-6"
)

summary_tokenizer = (
    AutoTokenizer.from_pretrained(
        SUMMARIZATION_MODEL_NAME
    )
)

summarization_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        SUMMARIZATION_MODEL_NAME
    )
)

print(
    "Summarization model loaded successfully."
)


# =========================================================
# 25. Review summarization function
# =========================================================

def summarize_review(raw_text):
    """
    Generate a short summary for reviews
    longer than 15 words.
    """

    if raw_text is None:

        raise ValueError(
            "Input text cannot be None."
        )

    raw_text = str(raw_text).strip()

    if not raw_text:

        raise ValueError(
            "Input text cannot be empty."
        )

    word_count = len(
        raw_text.split()
    )

    if word_count <= 15:

        return (
            "The text is too short "
            "to require summarization."
        )

    inputs = summary_tokenizer(
        raw_text,
        max_length=1024,
        return_tensors="pt",
        truncation=True
    )

    input_token_count = (
        inputs["input_ids"].shape[1]
    )

    maximum_summary_length = min(
        40,
        max(
            10,
            input_token_count // 2
        )
    )

    minimum_summary_length = min(
        10,
        maximum_summary_length - 1
    )

    summary_ids = summarization_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],

        max_new_tokens=maximum_summary_length,
        min_new_tokens=minimum_summary_length,

        no_repeat_ngram_size=2,
        num_beams=4,
        early_stopping=True
    )

    summary_text = summary_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    ).strip()

    if len(summary_text.split()) >= word_count:

        return raw_text

    return summary_text


# =========================================================
# 26. Combined analysis function
# =========================================================

def analyze_and_summarize_review(raw_text):
    """
    Apply:
    1. LSTM sentiment analysis
    2. DistilBART summarization
    """

    sentiment, confidence, probabilities = (
        predict_sentiment(raw_text)
    )

    summary = summarize_review(
        raw_text
    )

    result = {
        "original_text": raw_text,
        "summary": summary,
        "sentiment": sentiment,
        "confidence": confidence,
        "class_probabilities": probabilities
    }

    return result


# =========================================================
# 27. Save the LSTM model
# =========================================================

sentiment_model.save(
    "lstm_sentiment_model.keras"
)

print(
    "\nLSTM model saved as: "
    "lstm_sentiment_model.keras"
)


# Save the tokenizer
with open(
    "text_tokenizer.pkl",
    "wb"
) as tokenizer_file:

    pickle.dump(
        text_tokenizer,
        tokenizer_file
    )


# Save the label encoder
with open(
    "label_encoder.pkl",
    "wb"
) as encoder_file:

    pickle.dump(
        label_encoder,
        encoder_file
    )


print(
    "Tokenizer saved as: text_tokenizer.pkl"
)

print(
    "Label encoder saved as: label_encoder.pkl"
)


# =========================================================
# 28. Automated testing
# =========================================================

if __name__ == "__main__":

    food_review = (
        "I ordered this organic coffee last week. "
        "It tastes absolutely terrible and stale, "
        "and it gave me a bad stomach ache after drinking it. "
        "I do not recommend buying this brand at all."
    )

    result = analyze_and_summarize_review(
        food_review
    )

    print("\n")
    print("=" * 70)
    print("MODEL EVALUATION RESULTS")
    print("=" * 70)

    print(
        f"Test Loss: "
        f"{test_loss:.4f}"
    )

    print(
        f"Test Accuracy: "
        f"{test_accuracy:.4f} "
        f"({test_accuracy * 100:.2f}%)"
    )

    print(
        f"Calculated Accuracy: "
        f"{calculated_accuracy:.4f} "
        f"({calculated_accuracy * 100:.2f}%)"
    )

    print(
        "\nDetailed Classification Report:"
    )

    print(
        detailed_report
    )

    print(
        "Confusion Matrix:"
    )

    print(
        confusion_matrix_result
    )

    print("=" * 70)
    print("TEST REVIEW RESULT")
    print("=" * 70)

    print("\nOriginal Review:")
    print(
        result["original_text"]
    )

    print("\nGenerated Summary:")
    print(
        result["summary"]
    )

    print(
        "\nPredicted Sentiment:",
        result["sentiment"].upper()
    )

    print(
        f"Prediction Confidence: "
        f"{result['confidence'] * 100:.2f}%"
    )

    print(
        "\nProbabilities for all classes:"
    )

    for class_name, probability in result[
        "class_probabilities"
    ].items():

        print(
            f"{class_name.capitalize()}: "
            f"{probability * 100:.2f}%"
        )

    print("=" * 70)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Reading dataset...
Dataset loaded successfully.
Initial dataset shape: (100000, 10)
Available columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
Skipped corrupted rows: 22

First corrupted rows:
{'line': 2135, 'error': '\',\' expected after \'"\''}
{'line': 4292, 'error': 'Expected 10 columns, but found 5 columns.'}
{'line': 6417, 'error': '\',\' expected after \'"\''}
{'line': 10550, 'error': 'Expected 10 columns, but found 9 columns.'}
{'line': 14597, 'error': 'Expected 10 columns, but found 11 columns.'}
{'line': 18889, 'error': '\',\' expected after \'"\''}
{'line': 20758, 'error': 'Expected 10 columns, but found 21 columns.'}
{'line': 22800, 'error': '\',\' expected after \'"\''}
{'line': 30679, 'error': 'Expected 10 columns, but found 4 columns.'}
{'line': 32467, 'error': '\',\' expected after \'"\''}

Dataset shape after cleaning: (100000, 3)

Sentiment distribution:
Sentiment
positive    7

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (None, 111, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer (LSTM)               │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden_layer (Dense)      │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sentiment_output (Dense)        │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

Total params: 2,613,763 (9.97 MB)

Trainable params: 2,613,763 (9.97 MB)

... (2 outputs left out)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Summarization model loaded successfully.

LSTM model saved as: lstm_sentiment_model.keras
Tokenizer saved as: text_tokenizer.pkl
Label encoder saved as: label_encoder.pkl


[transformers] Both `max_new_tokens` (=20) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




MODEL EVALUATION RESULTS
Test Loss: 0.4843
Test Accuracy: 0.8329 (83.29%)
Calculated Accuracy: 0.8329 (83.30%)

Detailed Classification Report:
              precision    recall  f1-score   support

    negative     0.7241    0.7588    0.7410      2998
     neutral     0.3583    0.6640    0.4655      1604
    positive     0.9592    0.8650    0.9096     15398

    accuracy                         0.8329     20000
   macro avg     0.6805    0.7626    0.7054     20000
weighted avg     0.8757    0.8329    0.8487     20000

Confusion Matrix:
[[ 2275   450   273]
 [  245  1065   294]
 [  622  1457 13319]]
TEST REVIEW RESULT

Original Review:
I ordered this organic coffee last week. It tastes absolutely terrible and stale, and it gave me a bad stomach ache after drinking it. I do not recommend buying this brand at all.

Generated Summary:
Organic coffee tastes absolutely terrible and stale, and it gave me a bad stomach ache after

Predicted Sentiment: NEGATIVE
Prediction Confidence: 99.84%
